In [1]:
from openpyxl import load_workbook
import pandas as pd
from rapidfuzz import process, fuzz
import re

In [2]:
# --- Archivos ---
ARCHIVO_ENTRADA = 'Libro1.xlsx'
HOJA_DATOS = 'BASE DE DATOS'
ARCHIVO_SALIDA = 'interventoria_limpio.xlsx'

# --- Columnas del Excel ---
COL_NOMBRE = 'Nombres y Apellidos'
COL_CEDULA = 'No. Cedula de ciudadanía'
COL_CONTRATO = 'No. Contrato de Interventoría'
COL_ANIO = 'Año del Contrato de Interventoría'
COL_FECHA_INICIO = 'Fecha de Inicio del Contrato de Interventoría'
COL_FECHA_FIN = 'Fecha de Finalización del Contrato de Interventoría'
COL_OBJETO = 'Objeto del Contrato  de Interventoría'
COL_CARGO = 'Cargo que desempeña en el Contrato de Interventoría'
COL_PARTICIPACION = 'Participación TOTAL en el proyecto (%)'
COL_ESTADO = 'Contrato Estado'
COL_INTERVENTORIA = 'INTERVENTORÍA'
COL_UNIDAD = 'UNIDAD EJECUTORA'
COL_REGISTRO = 'REGISTRO'
COL_DEDICACION = 'Dedicación \n(hombre - mes)'
COL_PLAZO_TOTAL = 'Plazo TOTAL \ndel Proyecto en meses'
COL_NIT = 'NIT'
COL_OBSERVACIONES = 'OBSERVACIONES'

# --- Validación de filas ---
COLUMNAS_VACIAS = [COL_NOMBRE, COL_CEDULA]


# --- Valores por defecto ---
FECHA_VACIA = '0000-00-00'
VALOR_SIN_CARGO = 'SIN CARGO'
VALOR_SIN_DEFINIR = 'SIN DEFINIR'

# --- Normalización de nombres ---
TABLA_TILDES = str.maketrans({
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
})

# --- Meses ---
MESES = {
    'ENERO': '01', 'FEBRERO': '02', 'MARZO': '03', 'ABRIL': '04',
    'MAYO': '05', 'JUNIO': '06', 'JULIO': '07', 'AGOSTO': '08',
    'SEPTIEMBRE': '09', 'SETIEMBRE': '09', 'OCTUBRE': '10',
    'NOVIEMBRE': '11', 'DICIEMBRE': '12',
}

MESES_CORTO = {
    'ENE': '01', 'FEB': '02', 'MAR': '03', 'ABR': '04',
    'MAY': '05', 'JUN': '06', 'JUL': '07', 'AGO': '08',
    'SEP': '09', 'OCT': '10', 'NOV': '11', 'DIC': '12',
}





In [3]:
df = pd.read_excel(ARCHIVO_ENTRADA)
df.columns = df.columns.str.strip()

wb = load_workbook(ARCHIVO_ENTRADA)
ws = wb[HOJA_DATOS]

headers = [
    cell.value.strip() if isinstance(cell.value, str) else cell.value
    for cell in ws[1]
]


In [4]:
col_inicio = headers.index(COL_FECHA_INICIO) + 1
col_fin = headers.index(COL_FECHA_FIN) + 1

def tiene_color(celda):
    fill = celda.fill
    if fill is None:
        return False
    if fill.fill_type is None:
        return False
    color = fill.fgColor.rgb
    if color in [None, '00000000', 'FFFFFFFF']:
        return False
    return True

estados = []
for row in range(2, ws.max_row + 1):
    celda_inicio = ws.cell(row=row, column=col_inicio)
    celda_fin = ws.cell(row=row, column=col_fin)
    if tiene_color(celda_inicio) or tiene_color(celda_fin):
        estados.append('FINALIZO')
    else:
        estados.append('ACTIVO')

df[COL_ESTADO] = estados


In [5]:
df = df.drop(columns=[COL_PLAZO_TOTAL], errors='ignore')
df = df.drop(columns=[COL_REGISTRO], errors='ignore')
df = df.dropna(how='all')
df = df.dropna(
    subset=COLUMNAS_VACIAS,
    how='all'
)


In [6]:
df[COL_NOMBRE] = (
    df[COL_NOMBRE]
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.translate(TABLA_TILDES)
    .str.replace(r'[^A-ZÑ ]', '', regex=True)
)


In [7]:
# Convierte a numérico, lo inválido → NaN
df[COL_CEDULA] = (df[COL_CEDULA]
    .astype('string')
    .str.replace(r'\D', '', regex=True)
    .replace({'': pd.NA, 'None': pd.NA, 'nan': pd.NA})
)

In [8]:
# Definir la regex para detectar meses en contratos
RE_MES_EN_CONTRATO = r'(?:ENERO|FEBRERO|MARZO|ABRIL|MAYO|JUNIO|JULIO|AGOSTO|SEPTIEMBRE|OCTUBRE|NOVIEMBRE|DICIEMBRE)'


df[COL_CONTRATO] = (df[COL_CONTRATO]
    .astype('string')
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    # CAMBIO:"2.019" → "2019"
    .str.replace(r'\b((?:19|20)\d)\.\s*(\d{3})\b', r'\1\2', regex=True)

    # === FILTRAR FECHAS Y TEXTOS NO VÁLIDOS ===
    .mask(
        lambda x: (
            x.str.contains(RE_MES_EN_CONTRATO, na=False) |
            x.str.fullmatch(r'SIN\s+N[UÚ]MERO', na=False, flags=re.IGNORECASE)
        ),
        pd.NA
    )

    # === CASOS ESPECIALES (van PRIMERO prioridad) ===

    # 1. Año mal escrito de 5 dígitos: "1980-20204" → "1980"
    .str.replace(r'^(\d+)-20\d{3}$', r'\1', regex=True)

    # 2. Contratos con sufijo OS: "1723 - OS2" → "1723"
    .str.replace(r'^(\d+)\s*[-–]\s*OS\d+$', r'\1', regex=True, flags=re.IGNORECASE)

    # 3. Contratos LPA: "LPA NO. 001 DE 2023..." → "LPA NO. 001"
    .str.replace(r'^(LPA\s+NO\.?\s*\d+).*$', r'\1', regex=True, flags=re.IGNORECASE)

    # CAMBIO: casos 4 y 6 combinados → terciarios con año (con o sin "No.")
    # "No. 3-1-85910-002 de 2019" → "3185910002"
    # "3-1-86011-001 de 2019"→ "3186011001"
    .str.replace(r'(?:No\.?\s*)?3[-–]1[-–](\d+)[-–](\d+)\s+de\s+\d{4}', r'31\1\2', regex=True, flags=re.IGNORECASE)

    .str.replace(r'^(\d+)\s+de\s+[\d.]+$', r'\1', regex=True, flags=re.IGNORECASE)

    # 5. Terciarios con "No." sin año: "No. 3177258011" → "3177258011"
    .str.replace(r'No\.?\s*(\d{10,})', r'\1', regex=True)

    # 7. Terciarios sin año: "3-1-108847-001" → "31108847001"
    .str.replace(r'3[-–]1[-–](\d+)[-–](\d+)', r'31\1\2', regex=True)

    # 8. Contrato Fiducia con número concatenado: "Contrato Fiducia mercantil 31108847001" → "31108847001"
    .str.replace(r'.*?(\d{10,}).*', r'\1', regex=True)

    # CAMBIO: casos 9 y 10 combinados → sufijo ANT con o sin segundo número
    # "013597-0628881 ANT" → "013597"
    # "05202 ANT"→ "05202"
    .str.replace(r'^(\d+)(?:[-–]\d+)?\s+ANT$', r'\1', regex=True, flags=re.IGNORECASE)

    # === CASOS GENERALES ===

    # 11. Número con año completo (guion, slash): "984-2025", "984/2025" → "984"
    .str.replace(r'^(\d+)\s*[-–/]\s*(?:19|20)\d{2}$', r'\1', regex=True)

    # 12. Número con "DE" y año: "984 DE 2025" → "984"
    .str.replace(r'^(\d+)\s+DE\s+(?:19|20)\d{2}$', r'\1', regex=True, flags=re.IGNORECASE)

    # 13. Número con año corto: "984-25" → "984"
    .str.replace(r'^(\d+)\s*[-–]\s*\d{2}$', r'\1', regex=True)

    # 14. Número seguido de texto: "984 OXI SALENTO..." → "984"
    .str.replace(r'^(\d+)\s+[A-ZÁÉÍÓÚÑ].*', r'\1', regex=True)

    # 15. Eliminar año al final: "1022-2018" → "1022"
    .str.replace(r'-(?:19|20)\d{2}$', '', regex=True)
)

In [9]:
df[COL_ANIO] = (df[COL_ANIO]
    .astype('string')
    .str.extract(r'(19\d{2}|20\d{2})')[0]
    .astype('Int64')
)


In [10]:
# --- Regex para fechas ---
RE_SOLO_TEXTO = re.compile(r'[A-ZÁÉÍÓÚÑ\s,\.]+')
RE_ANIO_MALO = re.compile(r'^0(\d{3})-(\d{2})-(\d{2})$')
RE_FECHA_ISO = re.compile(r'(\d{4}-\d{2}-\d{2})')
RE_FECHA_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})/\(?(\d{4})')
RE_FECHA_DOBLE_SLASH = re.compile(r'(\d{1,2})/(\d{1,2})//(\d{2})')
RE_TEXTO_CON_FECHA = re.compile(r'(\d{1,2})\s+(?:DE\s+)?([A-ZÁÉÍÓÚÑ]+)\s+(?:DE|DEL)?\s+(\d{4})')
RE_FECHA_CORTA = re.compile(r'(\d{1,2})-([A-ZÁÉÍÓÚÑ]{3})-(\d{4})')
RE_MES_ANIO = re.compile(r'^([A-ZÁÉÍÓÚÑ]+)\s+DE\s+(\d{4})$')
RE_FECHA_USA = re.compile(r'^(\d{1,2})/(\d{1,2})/(\d{2})$')  # Formato americano M/D/YY


def convertir_fecha_texto(valor):
    if pd.isna(valor):
        return FECHA_VACIA

    valor = re.sub(r'\s+', ' ', str(valor).upper().strip())

    if valor in ('00/00/0000', '-', 'N/A', 'NA', ''):
        return FECHA_VACIA

    # Si solo tiene letras, espacios, comas o puntos, no es una fecha
    if RE_SOLO_TEXTO.fullmatch(valor):
        return FECHA_VACIA

    # Formato americano M/D/YY: 7/16/19 -> 2019-07-16
    m = RE_FECHA_USA.search(valor)
    if m:
        mes, dia, anio = m.groups()
        return f'20{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 0201-01-28 -> 2001-01-28
    m = RE_ANIO_MALO.search(valor)
    if m:
        anio, mes, dia = m.groups()
        return f'2{anio}-{mes}-{dia}'

    # 2016-07-28 00:00:00 -> 2016-07-28
    m = RE_FECHA_ISO.search(valor)
    if m:
        return m.group(1)

    # FECHA PROBABLE 08/12/2021 / REINICIO 08/08/2022 / 09/12/(2019
    m = RE_FECHA_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 17/03//21 -> 2021-03-17
    m = RE_FECHA_DOBLE_SLASH.search(valor)
    if m:
        dia, mes, anio = m.groups()
        return f'20{anio}-{mes.zfill(2)}-{dia.zfill(2)}'

    # 07 ENERO DE 2020 / 14 OCTUBRE DEL 2022 / 1 DE DICIEMBRE 2022
    m = RE_TEXTO_CON_FECHA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # ESTIMADO 6-FEB-2023
    m = RE_FECHA_CORTA.search(valor)
    if m:
        dia, mes_txt, anio = m.groups()
        mes = MESES_CORTO.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-{dia.zfill(2)}'

    # FEBRERO DE 2019 -> 2019-02-00
    m = RE_MES_ANIO.search(valor)
    if m:
        mes_txt, anio = m.groups()
        mes = MESES.get(mes_txt)
        if mes:
            return f'{anio}-{mes}-00'

    # Todo lo que no se pudo convertir queda como fecha por defecto
    return FECHA_VACIA


# === APLICAR CONVERSIÓN ===
df[COL_FECHA_INICIO] = df[COL_FECHA_INICIO].apply(convertir_fecha_texto)
df[COL_FECHA_FIN] = df[COL_FECHA_FIN].apply(convertir_fecha_texto)

# === NUEVA LÓGICA: Si COL_ESTADO es FINALIZO, fecha final = FECHA_VACIA ===
df.loc[
    df[COL_ESTADO].astype(str).str.upper().str.contains('FINALIZO', na=False),
    COL_FECHA_FIN] = FECHA_VACIA

In [11]:
### Objeto del Contrato

MIN_PALABRAS_OBJETO = 3

# Limpieza básica de la columna
df[COL_OBJETO] = (df[COL_OBJETO]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

# Función para obtener la descripción más completa

def descripcion_mas_completa(grupo):
    # Filtrar nulos y vacíos (usando .loc con lambda para mayor limpieza)
    descripciones = grupo.dropna().astype('string').str.strip()
    descripciones = descripciones.loc[lambda x: x != '']

    if descripciones.empty:
        return pd.NA

    # Retornar la descripción más larga
    return descripciones.loc[descripciones.str.len().idxmax()]


# Aplicar por grupo (Contrato + Año)
df[COL_OBJETO] = (df
    .groupby([COL_CONTRATO, COL_ANIO])[COL_OBJETO]
    .transform(descripcion_mas_completa)
)

# Contar palabras y dejar en blanco las inválidas

# Contar cuántas palabras tiene cada descripción
palabras_objeto = df[COL_OBJETO].fillna('').str.findall(r'\b\w+\b').str.len()

# dejamos la celda en blanco que no cumplan el minimo
df.loc[palabras_objeto < MIN_PALABRAS_OBJETO, COL_OBJETO] = pd.NA

In [12]:
# ==========================================
# TABLA DE TILDES
# ==========================================
TABLA_TILDES = str.maketrans({
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
})

# ==========================================
# PASO 0 — Normalizar texto
# ==========================================
def normalizar_texto(valor):
    if pd.isna(valor):
        return pd.NA

    texto = str(valor)

    # Quitar comillas, asteriscos, viñetas, saltos de línea
    texto = re.sub(r'["""\'*•\-–\n\r\t]', ' ', texto)
    # Quitar paréntesis con contenido numérico: "(1)", "(5)", etc.
    texto = re.sub(r'\(\d+\)', ' ', texto)
    # Mayúsculas
    texto = texto.upper()
    # Quitar tildes
    texto = texto.translate(TABLA_TILDES)
    # Quitar punto final suelto
    texto = re.sub(r'\.\s*$', '', texto)
    # Quitar caracteres no alfanuméricos al inicio y final
    texto = re.sub(r'^[^A-ZN0-9]+', '', texto)
    texto = re.sub(r'[^A-ZN0-9]+$', '', texto)
    # Espacios extra
    texto = re.sub(r'\s+', ' ', texto).strip()

    return texto if texto else pd.NA


# ==========================================
# PASO 1 — Filtrar basura
# ==========================================
RE_SOLO_NUMEROS = re.compile(r'^\d+$')
RE_FECHA = re.compile(r'^\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}$')
RE_FECHA_ESPACIOS = re.compile(r'^\d{4}\s+\d{2}\s+\d{2}\s+\d{2}:\d{2}:\d{2}$')
RE_TEXTO_LARGO = re.compile(r'^.{120,}$')   # párrafos largos → basura
RE_CODIGOS_RAROS = re.compile(
    r'^(?:SIG\d*|AUX\s+ADMON\s+VIAL\s*\d*|RESIDENTE\s+[A-Z]+\s+\d+|'
    r'ING\s*\.\s*RESIDENTE\s+\d+|RESIDENTE\s+\d+|'
    r'\d{1,2}:\d{2}|INSPECTOR\s+\d+|40)$'
)

def es_basura(valor):
    if pd.isna(valor):
        return True
    v = str(valor).strip()
    if not v:
        return True
    if RE_SOLO_NUMEROS.match(v):
        return True
    if RE_FECHA.match(v):
        return True
    if RE_TEXTO_LARGO.match(v):
        return True
    if RE_CODIGOS_RAROS.match(v):
        return True
    if RE_FECHA_ESPACIOS.match(v):
        return True
    return False


# ==========================================
# PASO 2 — Homologación exacta
# ==========================================
HOMOLOGAR_EXACTO = {
    # Abreviaciones
    r'\bING\.\b'          : 'INGENIERO',
    r'\bING\b'            : 'INGENIERO',
    r'\bESP\.\b'          : 'ESPECIALISTA',
    r'\bESP\b'            : 'ESPECIALISTA',
    r'\bPROF\.\b'         : 'PROFESIONAL',
    r'\bPROF\b'           : 'PROFESIONAL',
    r'\bDIR\b'            : 'DIRECTOR',
    r'\bRES\b'            : 'RESIDENTE',
    r'\bAUX\b'            : 'AUXILIAR',
    r'\bING\.\s*RESIDENTE\b'          : 'INGENIERO RESIDENTE',
    r'\bING\.\s*AUXILIAR\b'           : 'INGENIERO AUXILIAR',
    r'\bING\.\s*DIRECTOR\b'           : 'INGENIERO DIRECTOR',
    r'\bESP\.\s*EN\b'                 : 'ESPECIALISTA EN',
    r'\bESP\s+EN\b'                   : 'ESPECIALISTA EN',


    # Géneros femeninos
    r'\bINGENIERA\b'      : 'INGENIERO',
    r'\bDIRECTORA\b'      : 'DIRECTOR',
    r'\bAUDITORA\b'       : 'AUDITOR',
    r'\bINSPECTORA\b'     : 'INSPECTOR',
    r'\bABOGADA\b'        : 'ABOGADO',
    r'\bTOPOGRAFA\b'      : 'TOPOGRAFO',
    r'\bDIRECTRIZ\b'      : 'DIRECTOR',


    # Errores tipográficos frecuentes
    r'\bRECIDENTE\b'      : 'RESIDENTE',
    r'\bRESICENTE\b'      : 'RESIDENTE',
    r'\bRESIDNETE\b'      : 'RESIDENTE',
    r'\bRESIDENTR\b'      : 'RESIDENTE',
    r'\bRESIDENTEN\b'     : 'RESIDENTE',
    r'\bRESIDENTS\b'      : 'RESIDENTE',
    r'\bREISDENTE\b'                  : 'RESIDENTE',
    r'\bRESIDEENTE\b'                 : 'RESIDENTE',
    r'\bRESDENTE\b'                   : 'RESIDENTE',



    r'\bINGENIRO\b'       : 'INGENIERO',
    r'\bINGNIERO\b'       : 'INGENIERO',
    r'\bINGENIEOR\b'      : 'INGENIERO',
    r'\bINGENIEROA\b'     : 'INGENIERO',
    r'\bINGTENIERO\b'     : 'INGENIERO',
    r'\bINGENERO\b'       : 'INGENIERO',
    r'\bINGENEIRO\b'      : 'INGENIERO',
    r'\bINGEIERO\b'       : 'INGENIERO',
    r'\bINGENERIO\b'      : 'INGENIERO',
    r'\bINGENIERIO\b'     : 'INGENIERO',
    r'\bINGINEERO\b'      : 'INGENIERO',
    r'\bINGENOERO\b'      : 'INGENIERO',
    r'\bINGENIRIA\b'      : 'INGENIERIA',
    r'\bINGEINERO\b'                  : 'INGENIERO',
    r'\bINGENEERO\b'                  : 'INGENIERO',
    r'\bINGENIEROS\b'                 : 'INGENIERO',
    r'\bINGENIEERO\b'                 : 'INGENIERO',
    r'\bINGENINERO\b'                 : 'INGENIERO',
    r'\bINGENNIERO\b'                 : 'INGENIERO',
    r'\bINGENEIERO\b'                 : 'INGENIERO',
    r'\bINGIENERO\b'                  : 'INGENIERO',

    r'\bESPECILAISTA\b'   : 'ESPECIALISTA',
    r'\bESPECILISTA\b'    : 'ESPECIALISTA',
    r'\bESPECILSITA\b'    : 'ESPECIALISTA',
    r'\bESPECIALSITA\b'   : 'ESPECIALISTA',
    r'\bESPECIALISAT\b'   : 'ESPECIALISTA',
    r'\bESECIALISTA\b'    : 'ESPECIALISTA',
    r'\bEPECIALISTA\b'    : 'ESPECIALISTA',
    r'\bESPECAILISTA\b'   : 'ESPECIALISTA',
    r'\bESPECIALIATA\b'   : 'ESPECIALISTA',
    r'\bESPECIAISTA\b'    : 'ESPECIALISTA',
    r'\bESPECIALSTA\b'    : 'ESPECIALISTA',
    r'\bESPCIALISTA\b'    : 'ESPECIALISTA',
    r'\bESPECIALISTO\b'   : 'ESPECIALISTA',
    r'\bESPECILISAT\b'    : 'ESPECIALISTA',
    r'\bESPECIALISA\b'    : 'ESPECIALISTA',
    r'\bEEPECIALISTA\b'   : 'ESPECIALISTA',
    r'\bESPECIAALISTA\b'  : 'ESPECIALISTA',
    r'\bESPECIAASLISTA\b' : 'ESPECIALISTA',
    r'\bESPECIALISITA\b'  : 'ESPECIALISTA',
    r'\bESPECILAISITA\b'  : 'ESPECIALISTA',
    r'\bESPECIALSIATA\b'  : 'ESPECIALISTA',
    r'\bESPECAILSTAI\b'   : 'ESPECIALISTA',
    r'\bESPECIALSIT\b'    : 'ESPECIALISTA',
    r'\bESPECIASLISTA\b'  : 'ESPECIALISTA',
    r'\bESPACIALISTA\b'   : 'ESPECIALISTA',
    r'\bESPECILASTA\b'    : 'ESPECIALISTA',
    r'\bESPECIALISATA\b'  : 'ESPECIALISTA',
    r'\bESPCIALISTAS\b'   : 'ESPECIALISTA',
    r'\bESPECIALISDA\b'               : 'ESPECIALISTA',
    r'\bESPECIALISISTA\b'             : 'ESPECIALISTA',
    r'\bESPECIALISTAS\b'              : 'ESPECIALISTA',
    r'\bESPECISALISTA\b'              : 'ESPECIALISTA',
    r'\bESPECISALITA\b'               : 'ESPECIALISTA',
    r'\bESPECIALISTAE\b'              : 'ESPECIALISTA',
    r'\bESPECIISTA\b'                 : 'ESPECIALISTA',
    r'\bESPECIISALISTA\b'             : 'ESPECIALISTA',
    r'\bESPECIALIISTA\b'              : 'ESPECIALISTA',



    r'\bPROFESINAL\b'     : 'PROFESIONAL',
    r'\bPROFSIONAL\b'     : 'PROFESIONAL',
    r'\bPROFESONAL\b'     : 'PROFESIONAL',
    r'\bPROFECIONAL\b'    : 'PROFESIONAL',
    r'\bPROFESIONAL\b'    : 'PROFESIONAL',
    r'\bPROFESINOAL\b'    : 'PROFESIONAL',
    r'\bPROFESIOINAL\b'   : 'PROFESIONAL',
    r'\bPROFESIONLA\b'    : 'PROFESIONAL',
    r'\bPROFESIONALA\b'   : 'PROFESIONAL',
    r'\bPROFESIONALS\b'               : 'PROFESIONAL',
    r'\bPROFECSOINAL\b'               : 'PROFESIONAL',
    r'\bPROFESOINAL\b'                : 'PROFESIONAL',

    r'\bINTERVETORIA\b'   : 'INTERVENTORIA',
    r'\bINTERVENOTRIA\b'  : 'INTERVENTORIA',
    r'\bINTERVETNORIA\b'  : 'INTERVENTORIA',
    r'\bINTERVENTORO\b'   : 'INTERVENTORIA',
    r'\bINTERVENTION\b'   : 'INTERVENTORIA',
    r'\bINTERVEMTORIA\b'  : 'INTERVENTORIA',
    r'\bINTERVENTORIA\b'  : 'INTERVENTORIA',
    r'\bINTEVENTORA\b'    : 'INTERVENTORIA',
    r'\bINTERVENTOR\b'    : 'INTERVENTORIA',
    r'\bINTERNVENTORIA\b' : 'INTERVENTORIA',
    r'\bINTERVENTORIE\b'  : 'INTERVENTORIA',
    r'\bINTERVENTRIA\b'   : 'INTERVENTORIA',
    r'\bINTEREVENTORIA\b' : 'INTERVENTORIA',
    r'\bINTEVENTORIA\b'               : 'INTERVENTORIA',
    r'\bINTERVENTORIAS\b'             : 'INTERVENTORIA',
    r'\bINTEVNETORIA\b'               : 'INTERVENTORIA',
    r'\bINTEVENTRIA\b'                : 'INTERVENTORIA',
    r'\bINTERVENTOROIA\b'             : 'INTERVENTORIA',

    # Variantes de AUXILIAR
    r'\bAUXILAR\b'        : 'AUXILIAR',
    r'\bAUXLIAR\b'        : 'AUXILIAR',
    r'\bAUXILIARL\b'      : 'AUXILIAR',
    r'\bAUXILLAR\b'       : 'AUXILIAR',
    r'\bAUXIILAR\b'       : 'AUXILIAR',
    r'\bAUXILIAR\b'                   : 'AUXILIAR',

    # AUDITOR - todas las variantes van a AUDITOR DE CALIDAD
    r'\bAUDITOR\s+IUNTERNO\s+DE\s+LA\s+CALIDAD\b'   : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+INTERNO\s+DE\s+CALIDAD\b'         : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+INTERNA\s+DE\s+CALIDAD\b'         : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+DE\s+CALIDAD\s+INTERVENTORIA\b'   : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+DECALIDAD\b'                      : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+CALIDAD\b'                        : 'AUDITOR DE CALIDAD',
    r'\bAUDITOR\s+INTERNO\b'                        : 'AUDITOR DE CALIDAD',

    # Otras correcciones frecuentes en tus datos
    r'\bGEOTECNICA\b'     : 'GEOTECNIA',
    r'\bGEOTECIA\b'       : 'GEOTECNIA',
    r'\bGEOTECNA\b'       : 'GEOTECNIA',
    r'\bGEOTECNISTA\b'    : 'GEOTECNIA',
    r'\bGEOTECNO\b'       : 'GEOTECNIA',
    r'\bGEOTENCIA\b'      : 'GEOTECNIA',
    r'\bGEOTENCIO\b'      : 'GEOTECNIA',
    r'\bGEOCTENIA\b'      : 'GEOTECNIA',
    r'\bGEOTECINIA\b'     : 'GEOTECNIA',
    r'\bGEOTENIA\b'                   : 'GEOTECNIA',
    r'\bGEOTECTNIA\b'                 : 'GEOTECNIA',
    r'\bGEOTECNOIA\b'                 : 'GEOTECNIA',

    r'\bHIDRAULICO\b'     : 'HIDRAULICA',
    r'\bHIDRAUILCA\b'     : 'HIDRAULICA',
    r'\bHIDRAULCA\b'      : 'HIDRAULICA',
    r'\bHIDRAUCA\b'       : 'HIDRAULICA',
    r'\bHIDRUALICA\b'     : 'HIDRAULICA',
    r'\bHIFRAULICA\b'     : 'HIDRAULICA',
    r'\bHIDRAULICAE\b'                : 'HIDRAULICA',
    r'\bHIDRAUICA\b'                  : 'HIDRAULICA',

    r'\bPAVIMENTO\b'      : 'PAVIMENTOS',
    r'\bPAVIMIENTOS\b'    : 'PAVIMENTOS',
    r'\bPAVIMIENTO\b'     : 'PAVIMENTOS',
    r'\bPAVIMENTOS\b'                 : 'PAVIMENTOS',

    r'\bESTRUCTURA\b'     : 'ESTRUCTURAS',
    r'\bESTRUCURAS\b'     : 'ESTRUCTURAS',
    r'\bESTRUCTUAS\b'     : 'ESTRUCTURAS',
    r'\bESTRUCTRAL\b'     : 'ESTRUCTURAL',
    r'\bESTRCUTURAS\b'    : 'ESTRUCTURAS',
    r'\bESTRUVCTURAS\b'   : 'ESTRUCTURAS',
    r'\bESTRCUTURAL\b'    : 'ESTRUCTURAL',
    r'\bESTRUCRAS\b'                  : 'ESTRUCTURAS',
    r'\bESTRUCITURAS\b'               : 'ESTRUCTURAS',
    r'\bESTRUCTURRAL\b'               : 'ESTRUCTURAL',
    r'\bESTRUTUCURAS\b'               : 'ESTRUCTURAS',


    r'\bAMBIETAL\b'       : 'AMBIENTAL',
    r'\bAMBIENTA\b'       : 'AMBIENTAL',
    r'\bAMBIENTAL\b'      : 'AMBIENTAL',
    r'\bAMIBENTAL\b'      : 'AMBIENTAL',
    r'\bAMBIETNAL\b'      : 'AMBIENTAL',
    r'\bAMBINETAL\b'      : 'AMBIENTAL',
    r'\bAMBIENATL\b'      : 'AMBIENTAL',
    r'\bAMBINATAL\b'      : 'AMBIENTAL',
    r'\bAMBIENAL\b'       : 'AMBIENTAL',
    r'\bAMBINATL\b'                   : 'AMBIENTAL',
    r'\bAMBIENTALS\b'                 : 'AMBIENTAL',
    r'\bAMBIENTEL\b'                  : 'AMBIENTAL',

    r'\bGEOTECNIA\s+VIAL\b'          : 'GEOTECNIA',
    r'\bSOSTENIBILIDAD\b'             : 'SOSTENIBILIDAD',
    r'\bSOSTENIBILDAD\b'              : 'SOSTENIBILIDAD',
    r'\bSOSTENIBILIAD\b'              : 'SOSTENIBILIDAD',
    r'\bDISENO\b'                     : 'DISENO',
    r'\bHIDROLOGIA\b'                 : 'HIDROLOGIA',
    r'\bSOCIOAMBIENTAL\b' : 'SOCIOAMBIENTAL',
    r'\bDE EL\b'          : 'DEL',
    r'\bDE LA INTERVENTORIA\b' : 'DE INTERVENTORIA',
    r'\bAUXILIAR\s+\d+\s+INTERVENTORIA\b': 'AUXILIAR DE INTERVENTORIA',
    r'\bAUDITOR\s+DE\s+CALIDA\b'  : 'AUDITOR DE CALIDAD',
    r'\bADITOR\s+DE\s+CALIDAD\b'  : 'AUDITOR DE CALIDAD',
    r'\bAUXILIAR ADMINISTRATIVO DE INTERVENTORIA\b': 'AUXILIAR ADMINISTRATIVO',
    r'\bAUXILIAR DE INGENIERO\b': 'AUXILIAR DE INGENIERIA',
    r'\bABOGADO PREDIAL INTERVENTORIA\b': 'ABOGADO PREDIAL',
    r'\bAUXILIAR DE INGENIERIA PARA INTERVENTORIA NUMERO\b': 'AUXILIAR DE INGENIERIA',
    r'\bDIRECTOR CONSULTOR\b': 'DIRECTOR CONSULTORIA',
    r'\bCOORDINADOR CONSULTORIA\b': 'COORDINADOR DE CONSULTORIA',
    r'\bINGENIERO COORDINADORA\b': 'INGENIERO COORDINADOR',
    r'\bAUXILIAR DE INTERVENTORI\b': 'AUXILIAR DE INTERVENTORIA',
    r'\bDIBUJANTE\s+AUTOCAD\s+SIG\b': 'DIBUJANTE AUTOCAD Y SIG',
    r'\bDIBUJANTE\s+SIG\b': 'DIBUJANTE AUTOCAD Y SIG',
    r'\bDIRECTOR DE INTERVENTORIA ESPECIALISTA EN PAVIMENTOS\b': 'DIRECTOR DE INTERVENTORIA Y ESPECIALISTA EN PAVIMENTOS',
    r'\bDIRECTOR CONSULTORIA\b': 'DIRECTOR DE CONSULTORIA',
    r'\bESPECIALISTA EN HIDRAULICA E HIDROLOGIA DE INTERVENTORIA\b': 'ESPECIALISTA EN HIDRAULICA E HIDROLOGIA',
    r'\bTECNICO EN MANTENIMIENTO\b': 'TECNICO DE MANTENIMIENTO',
    r'\bRESIDENTE ADM VIAL\b': 'RESIDENTE ADMINISTRACION VIAL',
    r'\bDIRECTOR\s+INTERV\b'           : 'DIRECTOR DE INTERVENTORIA',
    r'\bEASPECIALISTA\b'               : 'ESPECIALISTA',
    r'\bESO\s+AMBIENTAL\b'             : 'ESPECIALISTA AMBIENTAL',
    r'\bESPECIALIASTA\b'               : 'ESPECIALISTA',
    r'\bVALUATORIO\b'                  : 'AVALUADOR',
    r'\bASEGURAMIENTO CALIDAD\b': 'AUDITOR DE CALIDAD',
    r'\bESPECIALISTA\s+CALIDAD\b': 'ESPECIALISTA EN CALIDAD',
    r'\bESPECIALISTA\s+DE\s+CALIDAD\b': 'ESPECIALISTA EN CALIDAD',
    r'\bESPECIALISTA\s+ASEG\.?\s*CALIDAD\b': 'ESPECIALISTA EN ASEGURAMIENTO DE CALIDAD',
    r'\bESPECIALISTA\s+ASEGURAMIENTO\s+DE\s+LA\s+CALIDAD\b': 'ESPECIALISTA EN ASEGURAMIENTO DE CALIDAD',
    r'\bESPC\.?\s+EN\s+DISENO\s+GEOMETRICO\b': 'ESPECIALISTA EN DISEÑO GEOMETRICO',
    r'\bESPECIALISTA\s+EN\s+DISEÑO\s+GEOMETRICOD\b': 'ESPECIALISTA EN DISEÑO GEOMETRICO',
    r'\bESPECIALISTA\s+EN\s+DISENO\s+GEOMETRICO\b': 'ESPECIALISTA EN DISEÑO GEOMETRICO',
    r'\bINGENIERO AUXILIAR DE INGENEIRIA\b': 'INGENIERO AUXILIAR DE INGENIERIA',
    r'\bINGENIERO AUXILIAR DE INETRVENTORIA\b': 'INGENIERO AUXILIAR DE INTERVENTORIA',
    r'\bINGENIERO AUXILIAR DE INTERVENTORIA\s+ADMINISTRACION\s+VIAL\b': 'INGENIERO AUXILIAR DE INTERVENTORIA',
    r'\bINGENIERO AUXILIAR DE INTERVENTORIA\s+DE\s+ADMINISTRADOR\s+VIAL\b': 'INGENIERO AUXILIAR DE INTERVENTORIA',
    r'ESPECIALISTA AMBIENTAL-SOSTENIBILIDAD': 'ESPECIALISTA AMBIENTAL Y/O SOSTENIBILIDAD',
    r'ESPECIALISTA AMBIENTAL INTERVENTORIA': 'ESPECIALISTA AMBIENTAL',
    r'\bESPAMBIENTAL\b': 'ESPECIALISTA AMBIENTAL',
}


# ==========================================
# PASO 3 — Quitar sufijos numéricos y de turno
# ==========================================
RE_SUFIJOS = re.compile(
    r'\s*(?:'
    r'NO\.?\s*\d+'          # No. 1, No.2
    r'|\(\d+\)'             # (1), (5)
    r'|\s+\d+$'             # " 1", " 2" al final
    r'|[-\s]+\d+$'          # "-1", " - 2" al final
    r'|[-–]\s*[A-Z]{1,3}\d+$'  # -AV1, -M1
    r'|\s+[A-Z]{1,2}\d+$'  # M1, T1, AV1
    r'|:\s*$'               # dos puntos sueltos al final
    r'|\.\s*$'              # punto suelto al final
    r'(INTERVENTORIA)\s+\d+\s*$'
    r')',
    re.IGNORECASE
)

def quitar_sufijos(valor):
    if pd.isna(valor):
        return valor
    # Aplicar varias veces por si hay sufijos anidados
    anterior = None
    resultado = str(valor)
    while resultado != anterior:
        anterior = resultado
        resultado = RE_SUFIJOS.sub('', resultado).strip()
    return resultado

# ==========================================
# PASO 4 — Lista de Cargos Válidos y Fuzzy
# ==========================================

CARGOS_VALIDOS = [
    "DIRECTOR DE INTERVENTORIA", "COORDINADOR DE INTERVENTORIA",
    "INGENIERO RESIDENTE", "INGENIERO RESIDENTE DE INTERVENTORIA",
    "RESIDENTE DE INTERVENTORIA", "RESIDENTE AUXILIAR DE INTERVENTORIA",
    "INGENIERO AUXILIAR DE INTERVENTORIA", "DIRECTOR DE OBRA",
    "RESIDENTE DE OBRA", "INGENIERO DE OBRA",
    "ESPECIALISTA EN ESTRUCTURAS", "ESPECIALISTA ESTRUCTURAL",
    "ESPECIALISTA EN GEOTECNIA", "ESPECIALISTA EN GEOLOGIA",
    "ESPECIALISTA EN PAVIMENTOS", "ESPECIALISTA EN VIAS",
    "ESPECIALISTA EN DISEÑO GEOMETRICO", "ESPECIALISTA EN HIDRAULICA",
    "ESPECIALISTA EN HIDROLOGIA", "ESPECIALISTA EN TRANSITO Y TRANSPORTE",
    "ESPECIALISTA EN SOSTENIBILIDAD", "ESPECIALISTA FORESTAL",
    "PROFESIONAL AMBIENTAL", "ESPECIALISTA AMBIENTAL",
    "PROFESIONAL SOCIAL", "TRABAJADOR SOCIAL", "RESIDENTE SOCIAL",
    "PROFESIONAL HSEQ", "COORDINADOR HSEQ", "PROFESIONAL SST",
    "TECNOLOGO SST", "INSPECTOR SST", "INSPECTOR SISO",
    "LABORATORISTA", "LABORATORISTA DE SUELOS", "JEFE DE LABORATORIO",
    "AUXILIAR DE LABORATORIO", "TOPOGRAFO", "AUXILIAR DE TOPOGRAFIA",
    "CADENERO", "DIBUJANTE", "DIBUJANTE CAD", "MODELADOR BIM",
    "PROGRAMADOR DE OBRA", "CONTROLADOR DE OBRA", "CONTROL DE CALIDAD",
    "PROFESIONAL DE CALIDAD", "PROFESIONAL PREDIAL",
    "PROFESIONAL JURIDICO", "PROFESIONAL VALUATORIO", "ABOGADO",
    "ARQUITECTO", "INGENIERO CIVIL", "INGENIERO AMBIENTAL",
    "INGENIERO FORESTAL", "INGENIERO SANITARIO", "INGENIERO HIDRAULICO",
    "INGENIERO ELECTRICO", "INGENIERO MECANICO", "INGENIERO INDUSTRIAL",
    "GEOLOGO", "BIOLOGO", "AUXILIAR ADMINISTRATIVO",
    "ASISTENTE ADMINISTRATIVO", "SECRETARIA", "CONTADOR",
    "REVISOR FISCAL", "APOYO SOCIAL", "AUXILIAR ADMINISTRATIVO VIAL",
    "ALMACENISTA", "BODEGUERO", "SUPERVISOR", "INSPECTOR DE OBRA",
    "CAPATAZ", "AUDITOR INTERNO", "BIOLOGO FAUNA Y FLORA",
    "AUXILIAR AMBIENTAL", "ESPECIALISTA EN EVALUACION ECONOMICA DE PROYECTOS"
    "ESPECIALISTA AMBIENTAL Y/O SOSTENIBILIDAD"
]

def homologar_cargo_fuzzy(valor, umbral=85):
    if pd.isna(valor) or str(valor).strip() == '':
        return valor
    resultado = process.extractOne(
        str(valor), CARGOS_VALIDOS,
        scorer=fuzz.token_sort_ratio,
        score_cutoff=umbral
    )
    if resultado:
        return resultado[0]
    return valor

# ==========================================
# EJECUCIÓN DEL PIPELINE
# ==========================================

# PASO 0: Normalizar texto
df[COL_CARGO] = df[COL_CARGO].apply(normalizar_texto)

# PASO 1: Filtrar basura (dejar en blanco, no borrar fila)
df.loc[df[COL_CARGO].apply(es_basura), COL_CARGO] = pd.NA
df[COL_CARGO] = df[COL_CARGO].str.replace(r'\.', '', regex=True)

# PASO 2: Homologación exacta
for pattern, replacement in HOMOLOGAR_EXACTO.items():
    df[COL_CARGO] = df[COL_CARGO].str.replace(pattern, replacement, regex=True)

# PASO 3: Quitar sufijos numéricos y de turno
df[COL_CARGO] = df[COL_CARGO].apply(quitar_sufijos)

# PASO 4: Fuzzy matching contra cargos válidos
df[COL_CARGO] = df[COL_CARGO].apply(homologar_cargo_fuzzy)

In [13]:
# ==========================================
# 1. Limpieza de texto para PARTICIPACIÓN y DEDICACIÓN
# ==========================================
for col in [COL_PARTICIPACION, COL_DEDICACION]:
    df[col] = (df[col]
        .astype(str)
        .str.replace('%', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.replace(' ', '', regex=False)
        .str.strip()
        .replace({'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA, '': pd.NA})
    )
    df[col] = df[col].apply(
        lambda x: pd.NA if pd.notna(x) and re.search(r'[a-zA-ZáéíóúÁÉÍÓÚñÑ]', str(x)) else x
    )

# ==========================================
# 2. Convertir a numérico
# ==========================================
df[COL_PARTICIPACION] = pd.to_numeric(df[COL_PARTICIPACION], errors='coerce')
df[COL_DEDICACION] = pd.to_numeric(df[COL_DEDICACION], errors='coerce')

# ==========================================
# 3. Normalizar DEDICACIÓN (Corregir decimales muy pequeños)
# ==========================================
def normalizar_dedicacion(valor):
    if pd.isna(valor):
        return valor
    if valor < 0.01:
        return valor * 100
    elif valor < 0.1:
        return valor * 10
    return valor

df[COL_DEDICACION] = df[COL_DEDICACION].apply(normalizar_dedicacion)

# ==========================================
# 4. Normalizar de 0-1 a 0-100
# ==========================================
for col in [COL_PARTICIPACION, COL_DEDICACION]:
    df.loc[(df[col] > 0) & (df[col] <= 1), col] = df[col] * 100

# ==========================================
# 5. Reemplazar valores > 100 en PARTICIPACION con DEDICACION
# ==========================================
mask = df[COL_PARTICIPACION] > 100
df.loc[mask, COL_PARTICIPACION] = df.loc[mask, COL_DEDICACION]

# Segunda pasada por si DEDICACION también estaba en escala 0-1
df.loc[(df[COL_PARTICIPACION] > 0) & (df[COL_PARTICIPACION] <= 1), COL_PARTICIPACION] = df[COL_PARTICIPACION] * 100

# ==========================================
# 6. Si DEDICACION > PARTICIPACION, usar DEDICACION
# ==========================================
mask = (df[COL_DEDICACION] > df[COL_PARTICIPACION]) | (df[COL_PARTICIPACION].isna() & df[COL_DEDICACION].notna())
df.loc[mask, COL_PARTICIPACION] = df.loc[mask, COL_DEDICACION]

# ==========================================
# 7. FINALIZO → vaciar PARTICIPACION
# ==========================================
df.loc[df['Contrato Estado'].str.upper().str.strip() == 'FINALIZO', COL_PARTICIPACION] = pd.NA

# ==========================================
# 8. Imputar por moda según cargo (ACTIVO con participacion < 10 o vacía)
# ==========================================
df[COL_PARTICIPACION] = df[COL_PARTICIPACION].apply(
    lambda x: float('nan') if x is pd.NA or str(x) in ('nan', 'None', '<NA>', '') else float(x)
    if pd.notna(x) else float('nan')
)
df[COL_PARTICIPACION] = df[COL_PARTICIPACION].astype('float64')

mask_imputar = (
    ((df[COL_PARTICIPACION] < 10) | df[COL_PARTICIPACION].isna()) &
    (df['Contrato Estado'].str.upper().str.strip() == 'ACTIVO')
)

moda_por_cargo = (
    df.loc[~mask_imputar]
    .groupby(COL_CARGO)[COL_PARTICIPACION]
    .agg(lambda x: float(x.dropna().mode()[0]) if not x.dropna().mode().empty else float('nan'))
)

df.loc[mask_imputar, COL_PARTICIPACION] = df.loc[mask_imputar, COL_CARGO].map(moda_por_cargo)
df[COL_PARTICIPACION] = df[COL_PARTICIPACION].round(2)
df[COL_PARTICIPACION] = df[COL_PARTICIPACION].clip(upper=100)
df = df.drop(columns=[COL_DEDICACION], errors='ignore')

In [14]:
df[COL_INTERVENTORIA] = (
    df[COL_INTERVENTORIA]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)

df.loc[
    df[COL_INTERVENTORIA].isna() |
    df[COL_INTERVENTORIA].astype('string').str.strip().eq('') |
    df[COL_INTERVENTORIA].astype('string').str.fullmatch(r'\d+(?:[,.]\d+)?', na=False) |
    df[COL_INTERVENTORIA].astype('string').str.contains(
        r'\d{1,2}[/\-]\d{1,3}[/\-]\d{2,4}|\d{4}-\d{1,2}-\d{1,2}|\d{1,2}\s+(?:DE\s+)?[A-ZÁÉÍÓÚÑ]+\s+(?:DE|DEL)\s+\d{4}',
        regex=True,
        na=False
    ) |
    df[COL_INTERVENTORIA].astype('string').str.contains(
        r'\b\d+(?:[,.]\d+)?\s*(?:MES|MESES|DIA|DIAS|AÑO|AÑOS)\b',
        regex=True,
        na=False
    ) |
    df[COL_INTERVENTORIA].astype('string').str.fullmatch(r'\d{6,12}-\d', na=False),
    COL_INTERVENTORIA
] = VALOR_SIN_DEFINIR


In [15]:
# ==========================================
# LIMPIEZA COL_NIT
# ==========================================

# PASO 1 — Limpiar fechas en formato dd/mm/yyyy
df[COL_NIT] = df[COL_NIT].astype(str).apply(
    lambda x: pd.NA if re.search(r'\d{1,2}/\d{1,2}/\d{4}', str(x)) else x
).replace({'nan': pd.NA, 'None': pd.NA, '<NA>': pd.NA})

# PASO 2 — Eliminar valores muy cortos
df[COL_NIT] = df[COL_NIT].apply(
    lambda x: pd.NA if pd.isna(x) or len(str(x).strip()) < 3 else x
)

# PASO 3 — Validar NIT y separar textos inválidos
def limpiar_nit(valor):
    if pd.isna(valor):
        return pd.NA, pd.NA

    texto = str(valor).strip()
    digitos = re.sub(r'\D', '', texto)
    letras = re.sub(r'[^a-zA-Z]', '', texto)

    # Fecha en formato datetime → vacío sin pasar a observaciones
    if re.match(r'^\d{4}-\d{2}-\d{2}', texto):
        return pd.NA, pd.NA

    # Más de 5 letras → texto libre, va a observaciones
    if len(letras) > 5:
        return pd.NA, texto

    # Pocos dígitos → observaciones
    if len(digitos) < 7:
        return pd.NA, texto

    return texto, pd.NA
df
nit_limpio, obs_nit = zip(*df[COL_NIT].apply(limpiar_nit))
df[COL_NIT] = list(nit_limpio)

# PASO 4 — Agregar textos inválidos a COL_OBSERVACIONES
df[COL_OBSERVACIONES] = df[COL_OBSERVACIONES].astype(str).replace({'nan': '', 'None': '', '<NA>': ''})
obs_nit = pd.Series(
    ['NIT inválido: ' + str(v) if pd.notna(v) else '' for v in obs_nit],
    index=df.index
)

df[COL_OBSERVACIONES] = df[COL_OBSERVACIONES].str.cat(
    obs_nit.where(obs_nit != '', other=''),
    sep=' | ',
    na_rep=''
).str.strip(' | ').replace({'': pd.NA})

In [16]:
# ==========================================
# LIMPIEZA DE LA COLUMNA: UNIDAD EJECUTORA
# ==========================================
COL_UNIDAD = 'UNIDAD EJECUTORA'

# Tabla para quitar tildes
TABLA_TILDES_UNIDAD = str.maketrans({
    'Á': 'A', 'É': 'E', 'Í': 'I', 'Ó': 'O', 'Ú': 'U',
})

# ==========================================
# PASO 1: Normalización de texto
# ==========================================
df[COL_UNIDAD] = (
    df[COL_UNIDAD]
    .astype('string')
    .str.upper()
    .str.strip()
    .str.strip('"')
    .str.replace(r'\s+', ' ', regex=True)
    .apply(lambda x: x.translate(TABLA_TILDES_UNIDAD) if pd.notna(x) else x)
)

# ==========================================
# PASO 2: Marcar valores inválidos como "SIN DEFINIR"
# ==========================================
# Creamos una versión limpia en memoria para las validaciones
unidad_limpia = df[COL_UNIDAD].astype('string').str.strip()

# Definimos qué se considera basura
es_basura_unidad = (
    pd.isna(df[COL_UNIDAD]) |                      # Nulos reales
    unidad_limpia.eq('') |                         # Cadenas vacías
    unidad_limpia.eq('|') |                        # Solo pipes
    unidad_limpia.eq('"') |                        # Solo comillas
    unidad_limpia.str.fullmatch(r'\d{4}-\d{2}-\d{2}(?: \d{2}:\d{2}:\d{2})?', na=False) # Fechas
)

df.loc[es_basura_unidad, COL_UNIDAD] = 'SIN DEFINIR'

# ==========================================
# PASO 3: Correcciones ortográficas y de tipeo
# ==========================================
CORRECCIONES_UNIDAD = {
    # DIRECCION TERRITORIAL
    'DIRECCION TERRIORIAL': 'DIRECCION TERRITORIAL',

    # SUBDIRECCION
    'SUBDIRECCCION DE GESTION INTEGRAL DE CARRETERAS NACIONALES': 'SUBDIRECCION DE GESTION INTEGRAL DE CARRETERAS NACIONALES',

    # GESTION INTEGRAL DE CARRETERAS NACIONALES
    'GESTION INTERGRAL DE CARRETERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTERGAL DE CARRETERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GETSION INTEGRAL DE CARRETERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION NTGRAL DE CARRETERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRETRAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRTERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRERAS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',
    'GESTION INTEGRAL DE CARRETERASS NACIONALES': 'GESTION INTEGRAL DE CARRETERAS NACIONALES',

    # RED NACIONAL DE CARRETERAS
    'RED NACIONAL DE CARRTERAS': 'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETARAS NACIONALES': 'RED NACIONAL DE CARRETERAS',
    'RED CARRETERAS NACIONALES': 'RED NACIONAL DE CARRETERAS',
    'RED DE CARRETERAS NACIONALES': 'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETERAS': 'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL CARRETERAS NACIONALES': 'RED NACIONAL DE CARRETERAS',
    'RED NACIONAL DE CARRETERAS NACIONALES': 'RED NACIONAL DE CARRETERAS',

    # GRANDES PROYECTOS
    'GRANDES PRPYECTOS': 'GRANDES PROYECTOS',
    'GRADES PROYECTOS': 'GRANDES PROYECTOS',
    'GRANSDES PROYECTOS': 'GRANDES PROYECTOS',

    # MODERNIZACION
    'MODRENIZACION': 'MODERNIZACION',

    # VIAS REGIONALES
    'RIAS REGIONALES': 'VIAS REGIONALES',
    'VIA REGIONALES': 'VIAS REGIONALES',
    'VAIS REGIONALES': 'VIAS REGIONALES',
    'VISAS REGIONALES': 'VIAS REGIONALES',
    'VIAS REFGIONALES': 'VIAS REGIONALES',
    'VIAAS REGIONALES': 'VIAS REGIONALES',

    # GRUPO REGIONAL
    'G. REGIONAL': 'GRUPO REGIONAL',
    'GRUPO REGIOMNAL': 'GRUPO REGIONAL',
}

# Aplicamos las correcciones
df[COL_UNIDAD] = df[COL_UNIDAD].replace(CORRECCIONES_UNIDAD, regex=False)

In [25]:
df.notnull().sum()

UNIDAD EJECUTORA                                       12949
Nombres y Apellidos                                    12949
No. Cedula de ciudadanía                               12949
No. Contrato de Interventoría                          12938
Año del Contrato de Interventoría                      12910
Fecha de Inicio del Contrato de Interventoría          12949
Fecha de Finalización del Contrato de Interventoría    12949
Objeto del Contrato  de Interventoría                  12899
Cargo que desempeña en el Contrato de Interventoría    12815
Participación TOTAL en el proyecto (%)                  4495
INTERVENTORÍA                                          12949
NIT                                                     9169
OBSERVACIONES                                          11080
Contrato Estado                                        12949
dtype: int64

In [17]:
df.to_excel(ARCHIVO_SALIDA, index=False)